Laberinto simple: Diseña un agente que llegue a la meta en una cuadrícula pequeña. Explica cómo explora caminos y cómo explota la ruta más corta.
Estados
Los estados representan la posición actual del agente dentro del laberinto.

Ejemplo:
(0,0)
(0,1)
(1,2)
Entorno
El entorno es un laberinto en forma de cuadrícula.

```text
A = Agente
M = Meta
X = Obstáculo

A . . .
X X . X
. . . X
X . . M
```



Acciones
El agente puede realizar:

Arriba Abajo Izquierda  Derecha


Política
La política define cómo el agente decide moverse.

En este proyecto se usa:

- Explorar caminos nuevos.
- Aprovechar rutas conocidas.



Recompensas

Acción Recompensa

 Llegar a la meta  +100 
 Movimiento válido  -1 
Chocar con pared  -10 


Exploración y Explotación

Exploración
El agente prueba caminos nuevos.

Explotación
El agente utiliza las rutas que ya aprendió.


In [21]:

import numpy as np
import random
import math


Crear el Laberinto

Usaremos:
0  espacio libre
1  obstáculo


In [22]:

laberinto = [
    [0, 0, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 1],
    [1, 0, 0, 0]
]

filas = len(laberinto)
columnas = len(laberinto[0])

# Posición inicial
inicio = (0, 0)

# Meta
meta = (3, 3)

# Acciones posibles
acciones = ['arriba', 'abajo', 'izquierda', 'derecha']

print("Laberinto cargado correctamente")


Laberinto cargado correctamente


Tabla Q

La tabla Q almacena:
Qué tan buena es cada acción.
Cuántas veces fue usada.


In [23]:
Q = {}
# Contador de veces que se usa cada acción
N = {}

for fila in range(filas):
    for columna in range(columnas):

        Q[(fila, columna)] = {}
        N[(fila, columna)] = {}

        for accion in acciones:
            Q[(fila, columna)][accion] = 0
            N[(fila, columna)][accion] = 1

print("Tabla Q creada")


Tabla Q creada


 entrenamiento


In [24]:

episodios = 500
alpha = 0.1
c = 2

print("Parámetros configurados")


Parámetros configurados



Función de Movimiento

Esta función:
- Mueve al agente.
- Detecta paredes.
- Entrega recompensas.


In [ ]:
def mover(estado, accion):

    fila, columna = estado

    # Movimiento arriba
    if accion == 'arriba':
        nueva_fila = fila - 1
        nueva_columna = columna

    # Movimiento abajo
    elif accion == 'abajo':
        nueva_fila = fila + 1
        nueva_columna = columna

    # Movimiento izquierda
    elif accion == 'izquierda':
        nueva_fila = fila
        nueva_columna = columna - 1

    # Movimiento derecha
    else:
        nueva_fila = fila
        nueva_columna = columna + 1

    # Verificar límites del tablero

    if (
        nueva_fila < 0 or
        nueva_fila >= filas or
        nueva_columna < 0 or
        nueva_columna >= columnas
    ):
        return estado, -10, False

    # Verificar obstáculos

    if laberinto[nueva_fila][nueva_columna] == 1:
        return estado, -10, False

    nuevo_estado = (nueva_fila, nueva_columna)

   
    # Verificar si llegó a la meta

    if nuevo_estado == meta:
        return nuevo_estado, 100, True

    # Movimiento normal
    return nuevo_estado, -1, False


Entrenamiento del Agente

Aquí el agente:
- Explora
- Aprende
- Actualiza valores Q


In [ ]:
for episodio in range(episodios):

    estado = inicio
    terminado = False
    paso = 1

    while not terminado:
      
        # SELECCIÓN DE ACCIÓN CON UCB

        mejor_valor = -999999

        for accion in acciones:

            valor_ucb = (
                Q[estado][accion]
                +
                c * math.sqrt(
                    math.log(paso + 1)
                    /
                    N[estado][accion]
                )
            )

            if valor_ucb > mejor_valor:
                mejor_valor = valor_ucb
                accion_elegida = accion

        # Aumentar contador
        N[estado][accion_elegida] += 1

        # Ejecutar acción
        nuevo_estado, recompensa, terminado = mover(
            estado,
            accion_elegida
        )

        # Mejor valor futuro
        mejor_valor_futuro = max(
            Q[nuevo_estado].values()
        )

 
        # ACTUALIZAR TABLA Q

        Q[estado][accion_elegida] = (
            Q[estado][accion_elegida]
            +
            alpha * (
                recompensa
                +
                gamma * mejor_valor_futuro
                -
                Q[estado][accion_elegida]
            )
        )

        # Actualizar estado
        estado = nuevo_estado

        paso += 1


Probar el Agente

Ahora veremos la ruta aprendida.


In [27]:
estado = inicio

ruta = [estado]

terminado = False

while not terminado:

    # Elegir mejor acción aprendida
    mejor_accion = max(
        Q[estado],
        key=Q[estado].get
    )

    # Mover agente
    estado, recompensa, terminado = mover(
        estado,
        mejor_accion
    )

    ruta.append(estado)

    # Seguridad para evitar bucles
    if len(ruta) > 20:
        break

print("Ruta aprendida:")
print(path)


Ruta aprendida:
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2), (3, 3)]
